In [ ]:
!pip install --upgrade pip -q
!pip install ultralytics ipywidgets --upgrade  -q
!pip install torch --index-url https://download.pytorch.org/whl/cu130 -q

In [ ]:
import os
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Paths
DATASET_PATH = Path("/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset")
# DATASET_PATH = "/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset"
OG_YAML = DATASET_PATH / "data.yaml"
DATA_YAML = "/kaggle/working/data_fixed.yaml"

WEIGHTS_DIR = Path("weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

In [ ]:
with open(OG_YAML, "r") as f:
    data = yaml.safe_load(f)

data["path"] = str(DATASET_PATH)

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Fixed YAML written to", DATA_YAML)

In [ ]:
# Training hyperparameters
CONFIG = {
    "model": "yolo11n.pt",  # Options: yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "workers": 8,
    "patience": 50, # Early stopping patience
    "save": True,
    "save_period": 10, # Save checkpoint every N epochs
    "cache": False, # Cache images for faster training (use True if enough RAM)
    "optimizer": "auto", # Options: SGD, Adam, AdamW, NAdam, RAdam, RMSProp, auto
    "lr0": 0.01, # Initial learning rate
    "lrf": 0.01, # Final learning rate (lr0 * lrf)
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,  # Box loss gain
    "cls": 0.5,  # Class loss gain
    "dfl": 1.5,  # DFL loss gain
    "hsv_h": 0.015,  # HSV-Hue augmentation
    "hsv_s": 0.7,  # HSV-Saturation augmentation
    "hsv_v": 0.4,  # HSV-Value augmentation
    "degrees": 0.0,  # Rotation augmentation
    "translate": 0.1,  # Translation augmentation
    "scale": 0.5,  # Scale augmentation
    "shear": 0.0,  # Shear augmentation
    "perspective": 0.0,  # Perspective augmentation
    "flipud": 0.0,  # Vertical flip probability
    "fliplr": 0.5,  # Horizontal flip probability
    "mosaic": 1.0,  # Mosaic augmentation probability
    "mixup": 0.0,  # Mixup augmentation probability
    "copy_paste": 0.0,  # Copy-paste augmentation probability
}

In [ ]:
print("Configuration:")
for key, value in CONFIG.items():
    print(f"{key}: {value}")

In [ ]:
print(f"\nVerifying dataset at: {DATA_YAML}")

# Load and display dataset info
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"Number of classes: {data_config['nc']}")
print(f"Train path: {data_config['train']}")
print(f"Val path: {data_config['val']}")
print(f"Test path: {data_config['test']}")
print(f"\nFirst 10 classes: {data_config['names'][:10]}")
print(f"Last 10 classes: {data_config['names'][-10:]}")

# Check if directories exist
train_images = Path(data_config['path']) / data_config['train']
val_images = Path(data_config['path']) / data_config['val']
test_images = Path(data_config['path']) / data_config['test']

print(f"\nTrain images exist: {train_images.exists()}")
print(f"Val images exist: {val_images.exists()}")
print(f"Test images exist: {test_images.exists()}")

In [ ]:
print(f"\nInitializing YOLOv11 model: {CONFIG['model']}")

# Load pretrained model
model = YOLO(CONFIG['model'])

# Display model info
print(f"Model loaded successfully")
print(f"Model type: {type(model)}")

In [ ]:
print("\n" + "="*80)
print("Starting Training")
print("="*80 + "\n")

# Train the model
results = model.train(
    data=str(DATA_YAML),
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    device=CONFIG['device'],
    workers=CONFIG['workers'],
    patience=CONFIG['patience'],
    save=CONFIG['save'],
    save_period=CONFIG['save_period'],
    cache=CONFIG['cache'],
    optimizer=CONFIG['optimizer'],
    lr0=CONFIG['lr0'],
    lrf=CONFIG['lrf'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
    warmup_epochs=CONFIG['warmup_epochs'],
    warmup_momentum=CONFIG['warmup_momentum'],
    warmup_bias_lr=CONFIG['warmup_bias_lr'],
    box=CONFIG['box'],
    cls=CONFIG['cls'],
    dfl=CONFIG['dfl'],
    hsv_h=CONFIG['hsv_h'],
    hsv_s=CONFIG['hsv_s'],
    hsv_v=CONFIG['hsv_v'],
    degrees=CONFIG['degrees'],
    translate=CONFIG['translate'],
    scale=CONFIG['scale'],
    shear=CONFIG['shear'],
    perspective=CONFIG['perspective'],
    flipud=CONFIG['flipud'],
    fliplr=CONFIG['fliplr'],
    mosaic=CONFIG['mosaic'],
    mixup=CONFIG['mixup'],
    copy_paste=CONFIG['copy_paste'],
    project=str(WEIGHTS_DIR),
    name='ingredient_detection',
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

print("\n" + "="*80)
print("Training Complete!")
print("="*80)

In [ ]:
print("\n" + "="*80)
print("Running Validation")
print("="*80 + "\n")

# Validate the model
metrics = model.val()

print(f"\nValidation Results:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

In [ ]:
print("\n" + "="*80)
print("Exporting Model")
print("="*80 + "\n")

# Export to different formats
# Uncomment the formats you need

# ONNX format (recommended for deployment)
# model.export(format='onnx')

# TensorRT format (for NVIDIA GPUs)
# model.export(format='engine')

# TorchScript format
# model.export(format='torchscript')

# CoreML format (for iOS)
# model.export(format='coreml')
model.save('best.pt')
# print("Model export skipped. Uncomment desired formats in Cell 7 to export.")

In [ ]:
print("\n" + "="*80)
print("Test Inference")
print("="*80 + "\n")

# Test on a sample image
# Uncomment to run inference on test images

# test_image = "/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/test/images/beef jerky_image_21.jpg"
test_image = "/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/test/images/banana blossom_image_0.jpg"
results = model.predict(
    source=test_image,
    conf=0.25, # Confidence threshold
    iou=0.45, # NMS IoU threshold
    save=True, # Save results
    show=False, # Show results
)

for result in results:
    boxes = result.boxes
    print(f"Detected {len(boxes)} objects")
    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        print(f"  Class: {data_config['names'][cls]}, Confidence: {conf:.2f}")



In [ ]:
# ============================================================================
# Cell 9: Summary
# ============================================================================

print("\n" + "="*80)
print("Training Summary")
print("="*80)
print(f"\nModel: {CONFIG['model']}")
print(f"Dataset: {DATA_YAML}")
print(f"Classes: {data_config['nc']}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Image size: {CONFIG['imgsz']}")
print(f"Batch size: {CONFIG['batch']}")
print(f"Device: {CONFIG['device']}")
print(f"\nBest weights saved to: {WEIGHTS_DIR / 'ingredient_detection' / 'weights' / 'best.pt'}")
print(f"Last weights saved to: {WEIGHTS_DIR / 'ingredient_detection' / 'weights' / 'last.pt'}")
print("\nTraining complete! Check the weights directory for results.")
print("="*80 + "\n")